# CrisisSense-LLM — Inference
Run this after training. Generates predictions for each checkpoint + zero-shot baseline.

Make sure your Google Drive is mounted and contains:
- `disaster_llmtest_v2.jsonl`
- `disaster_llm/outputs/qwen25_7b_lora/checkpoint-XXXX` (from training)

In [1]:
# STEP 1 — Install dependencies
!pip install -q torch transformers peft accelerate sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.1 MB/s eta 0:00:00


In [2]:
# STEP 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# STEP 3 — Paths config
# Change DRIVE_ROOT if your folder is named differently
import os

DRIVE_ROOT   = "/content/drive/MyDrive/disaster_llm"
BASE_MODEL   = "Qwen/Qwen2.5-7B-Instruct"
TEST_FILE    = f"{DRIVE_ROOT}/test_v2.jsonl"
CKPT_DIR     = f"{DRIVE_ROOT}/outputs/qwen25_7b_lora"
PRED_DIR     = f"{DRIVE_ROOT}/outputs/predictions"
MAX_REGEN    = 5      # max regeneration attempts per sample
LIMIT        = None  

os.makedirs(PRED_DIR, exist_ok=True)

# List available checkpoints
checkpoints = sorted([
    d for d in os.listdir(CKPT_DIR)
    if d.startswith("checkpoint-")
])
print("Available checkpoints:")
for c in checkpoints:
    print(" ", c)
print("\nTest file exists:", os.path.exists(TEST_FILE))

Available checkpoints:
  checkpoint-1604
  checkpoint-3208
  checkpoint-4812
  checkpoint-6416
  checkpoint-6418

Test file exists: True


In [4]:
# STEP 4 — Parsing functions (Section 3.4 of paper)
import re
import json

KEY_PATTERNS = {
    "event_type": [
        r'"event\s*type"\s*:\s*"([^"]*)"',
        r'"Event\s*type"\s*:\s*"([^"]*)"',
        r'event\s*type\s*[:=]\s*"?([A-Za-z_ ]+)"?',
    ],
    "useful": [
        r'"useful"\s*:\s*"?(True|False|true|false)"?',
        r'"Useful"\s*:\s*"?(True|False|true|false)"?',
        r'useful\s*[:=]\s*"?(True|False|true|false)"?',
    ],
    "humanitarian_aid_type": [
        r'"humanitarian\s*aid\s*type"\s*:\s*"([^"]*)"',
        r'"Humanitarian\s*aid\s*type"\s*:\s*"([^"]*)"',
        r'humanitarian\s*aid\s*type\s*[:=]\s*"?([A-Za-z_ ]+)"?',
    ],
}

def extract_field(text, field):
    for pattern in KEY_PATTERNS[field]:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            val = m.group(1).strip()
            if val:
                return val
    return "None"

def normalize_useful(val):
    if val.lower() == "true":  return "True"
    if val.lower() == "false": return "False"
    return "None"

def parse_response(text):
    if "Response:" in text:
        text = text.split("Response:")[-1]
    elif "### Response:" in text:
        text = text.split("### Response:")[-1]
    return {
        "Event type": extract_field(text, "event_type"),
        "Useful": normalize_useful(extract_field(text, "useful")),
        "Humanitarian aid type": extract_field(text, "humanitarian_aid_type"),
    }

def has_none(parsed):
    return any(v == "None" for v in parsed.values())

def load_template4_samples(test_file, limit=None):
    samples = []
    with open(test_file, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            if obj.get("template") == 4:
                samples.append(obj)
    if limit:
        samples = samples[:limit]
    return samples

print("Parsing functions loaded.")

Parsing functions loaded.


In [5]:
# STEP 5 — Model loading function
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

def load_model(base_model, model_path=None):
    print(f"Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"Loading base model (4-bit to save memory)...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    if model_path and model_path.lower() != "none":
        print(f"Loading LoRA adapter from {model_path}...")
        model = PeftModel.from_pretrained(model, model_path)

    model.eval()
    print("Model ready.")
    return tokenizer, model

print("Model loading function ready.")

Model loading function ready.


In [6]:
# STEP 6 — Generation function
@torch.no_grad()
def generate(tokenizer, model, prompt, max_new_tokens=150, do_sample=False, temperature=0.8):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    gen_kwargs = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.pad_token_id)
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen_kwargs.update(do_sample=False)
    output_ids = model.generate(**inputs, **gen_kwargs)
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print("Generation function ready.")

Generation function ready.


In [7]:
# STEP 7 — Main inference loop function
def run_inference(tokenizer, model, test_file, out_file, max_regen=5, limit=None):
    samples = load_template4_samples(test_file, limit)
    print(f"Running inference on {len(samples):,} samples...")

    os.makedirs(os.path.dirname(out_file), exist_ok=True)
    n_invalid = 0

    with open(out_file, "w", encoding="utf-8") as f_out:
        for i, sample in enumerate(samples):
            instruction  = sample["instruction"]
            ground_truth = sample["response"]
            attempts = []
            parsed   = None

            for attempt in range(max_regen):
                do_sample = attempt > 0
                raw    = generate(tokenizer, model, instruction,
                                  max_new_tokens=150, do_sample=do_sample)
                parsed = parse_response(raw)
                attempts.append({"raw": raw, "parsed": parsed})
                if not has_none(parsed):
                    break

            if parsed is None or has_none(parsed):
                n_invalid += 1

            f_out.write(json.dumps({
                "index": i,
                "ground_truth": ground_truth,
                "final_prediction": parsed,
                "n_attempts": len(attempts),
                "attempts": attempts,
            }, ensure_ascii=False) + "\n")

            if (i + 1) % 25 == 0:
                print(f"  {i+1}/{len(samples)} done ({n_invalid} invalid so far)")

    print(f"\nDone! Saved to {out_file}")
    print(f"Invalid after regen: {n_invalid}/{len(samples)} ({100*n_invalid/max(1,len(samples)):.1f}%)")
    return n_invalid

print("Inference loop function ready.")

Inference loop function ready.


## RUN A: Zero-shot baseline (no LoRA adapter)
This is Section 4.1 of the paper — base Qwen2.5 with no fine-tuning.

In [9]:
LIMIT = 500

In [ ]:
# RUN A — Zero-shot baseline
tokenizer, model = load_model(BASE_MODEL, model_path=None)

run_inference(
    tokenizer,
    model,
    test_file=TEST_FILE,
    out_file=f"{PRED_DIR}/zero_shot.jsonl",
    max_regen=MAX_REGEN,
    limit=LIMIT,
)

# Free memory before loading next model
import gc

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

KeyboardInterrupt: 

## RUN B: Fine-tuned checkpoints (Sections 4.2 and 4.3)
Runs inference for each checkpoint saved during training.

In [ ]:
# RUN B —
import gc
import torch


best_checkpoints = ["checkpoint-4812", "checkpoint-6416"]

for ckpt in best_checkpoints:
    ckpt_path = f"{CKPT_DIR}/{ckpt}"
    out_file  = f"{PRED_DIR}/{ckpt}.jsonl"

    if not os.path.exists(ckpt_path):
        print(f"Skipping {ckpt} - not found")
        continue

    print(f"\n{'='*50}")
    print(f"Running: {ckpt}")
    print(f"{'='*50}")

    tokenizer, model = load_model(BASE_MODEL, model_path=ckpt_path)

    run_inference(
        tokenizer, model,
        test_file=TEST_FILE,
        out_file=out_file,
        max_regen=MAX_REGEN,
        limit=LIMIT,
    )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Memory cleared. Moving to next checkpoint...")

print("\nAll done!")
print(f"Prediction files saved in: {PRED_DIR}")


Running: checkpoint-4812
Loading tokenizer...
Loading base model (4-bit to save memory)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading LoRA adapter from /content/drive/MyDrive/disaster_llm/outputs/qwen25_7b_lora/checkpoint-4812...
Model ready.
Running inference on 500 samples...
  25/500 done (0 invalid so far)
  50/500 done (0 invalid so far)
  75/500 done (0 invalid so far)
  100/500 done (0 invalid so far)
  125/500 done (0 invalid so far)
  150/500 done (0 invalid so far)
  175/500 done (0 invalid so far)
  200/500 done (0 invalid so far)
  225/500 done (0 invalid so far)
  250/500 done (0 invalid so far)
  275/500 done (0 invalid so far)
  300/500 done (0 invalid so far)
  325/500 done (0 invalid so far)
  350/500 done (0 invalid so far)
  375/500 done (0 invalid so far)
  400/500 done (0 invalid so far)
  425/500 done (0 invalid so far)
  450/500 done (0 invalid so far)
  475/500 done (0 invalid so far)
  500/500 done (0 invalid so far)

Done! Saved to /content/drive/MyDrive/disaster_llm/outputs/predictions/checkpoint-4812.jsonl
Invalid after regen: 0/500 (0.0%)
Memory cleared. Moving to next checkpoint.

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading LoRA adapter from /content/drive/MyDrive/disaster_llm/outputs/qwen25_7b_lora/checkpoint-6416...
Model ready.
Running inference on 500 samples...
  25/500 done (0 invalid so far)
  50/500 done (0 invalid so far)
  75/500 done (0 invalid so far)
  100/500 done (0 invalid so far)
  125/500 done (0 invalid so far)
  150/500 done (0 invalid so far)
  175/500 done (0 invalid so far)
  200/500 done (0 invalid so far)
  225/500 done (0 invalid so far)
  250/500 done (0 invalid so far)
  275/500 done (0 invalid so far)
  300/500 done (0 invalid so far)
  325/500 done (1 invalid so far)
  350/500 done (1 invalid so far)
  375/500 done (1 invalid so far)
  400/500 done (1 invalid so far)
  425/500 done (1 invalid so far)
  450/500 done (1 invalid so far)
  475/500 done (1 invalid so far)
  500/500 done (1 invalid so far)

Done! Saved to /content/drive/MyDrive/disaster_llm/outputs/predictions/checkpoint-6416.jsonl
Invalid after regen: 1/500 (0.2%)
Memory cleared. Moving to next checkpoint.

In [10]:
# RUN B —
import gc
import torch


best_checkpoints = ["checkpoint-1604", "checkpoint-3208"]

for ckpt in best_checkpoints:
    ckpt_path = f"{CKPT_DIR}/{ckpt}"
    out_file  = f"{PRED_DIR}/{ckpt}.jsonl"

    if not os.path.exists(ckpt_path):
        print(f"Skipping {ckpt} - not found")
        continue

    print(f"\n{'='*50}")
    print(f"Running: {ckpt}")
    print(f"{'='*50}")

    tokenizer, model = load_model(BASE_MODEL, model_path=ckpt_path)

    run_inference(
        tokenizer, model,
        test_file=TEST_FILE,
        out_file=out_file,
        max_regen=MAX_REGEN,
        limit=LIMIT,
    )

    del model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print(f"Memory cleared. Moving to next checkpoint...")

print("\nAll done!")
print(f"Prediction files saved in: {PRED_DIR}")


Running: checkpoint-1604
Loading tokenizer...
Loading base model (4-bit to save memory)...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loading LoRA adapter from /content/drive/MyDrive/disaster_llm/outputs/qwen25_7b_lora/checkpoint-1604...
Model ready.
Running inference on 500 samples...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  25/500 done (0 invalid so far)
  50/500 done (0 invalid so far)
  75/500 done (0 invalid so far)
  100/500 done (0 invalid so far)
  125/500 done (0 invalid so far)
  150/500 done (0 invalid so far)
  175/500 done (0 invalid so far)
  200/500 done (1 invalid so far)
  225/500 done (1 invalid so far)
  250/500 done (2 invalid so far)
  275/500 done (2 invalid so far)
  300/500 done (2 invalid so far)
  325/500 done (2 invalid so far)
  350/500 done (2 invalid so far)
  375/500 done (2 invalid so far)
  400/500 done (2 invalid so far)
  425/500 done (2 invalid so far)
  450/500 done (2 invalid so far)
  475/500 done (3 invalid so far)
  500/500 done (3 invalid so far)

Done! Saved to /content/drive/MyDrive/disaster_llm/outputs/predictions/checkpoint-1604.jsonl
Invalid after regen: 3/500 (0.6%)
Memory cleared. Moving to next checkpoint...

Running: checkpoint-3208
Loading tokenizer...
Loading base model (4-bit to save memory)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading LoRA adapter from /content/drive/MyDrive/disaster_llm/outputs/qwen25_7b_lora/checkpoint-3208...
Model ready.
Running inference on 500 samples...
  25/500 done (0 invalid so far)
  50/500 done (0 invalid so far)
  75/500 done (0 invalid so far)
  100/500 done (1 invalid so far)
  125/500 done (1 invalid so far)
  150/500 done (1 invalid so far)
  175/500 done (1 invalid so far)
  200/500 done (1 invalid so far)
  225/500 done (1 invalid so far)
  250/500 done (1 invalid so far)
  275/500 done (1 invalid so far)
  300/500 done (1 invalid so far)
  325/500 done (1 invalid so far)
  350/500 done (1 invalid so far)
  375/500 done (1 invalid so far)
  400/500 done (1 invalid so far)
  425/500 done (1 invalid so far)
  450/500 done (1 invalid so far)
  475/500 done (1 invalid so far)
  500/500 done (1 invalid so far)

Done! Saved to /content/drive/MyDrive/disaster_llm/outputs/predictions/checkpoint-3208.jsonl
Invalid after regen: 1/500 (0.2%)
Memory cleared. Moving to next checkpoint.

## Quick accuracy check (without evaluate_results.py)
Run this after inference to get a quick accuracy number for each prediction file.

In [11]:
# Quick accuracy check
FIELDS = ["Event type", "Useful", "Humanitarian aid type"]

def quick_accuracy(pred_file):
    records = []
    with open(pred_file, "r", encoding="utf-8") as f:
        for line in f:
            records.append(json.loads(line))

    n = len(records)
    if n == 0:
        return

    overall = 0
    field_correct = {f: 0 for f in FIELDS}

    for r in records:
        gt   = r["ground_truth"]
        pred = r["final_prediction"]
        matches = {}
        for f in FIELDS:
            gt_v   = str(gt.get(f,   "None")).strip().lower()
            pred_v = str(pred.get(f, "None")).strip().lower()
            matches[f] = (gt_v == pred_v) and (pred_v != "none")
            if matches[f]:
                field_correct[f] += 1
        if all(matches[f] for f in FIELDS):
            overall += 1

    name = os.path.basename(pred_file)
    print(f"\n--- {name} ---")
    print(f"  Overall accuracy:         {overall/n*100:.2f}%")
    print(f"  Event type accuracy:      {field_correct['Event type']/n*100:.2f}%")
    print(f"  Informativeness accuracy: {field_correct['Useful']/n*100:.2f}%")
    print(f"  Humanitarian accuracy:    {field_correct['Humanitarian aid type']/n*100:.2f}%")
    print(f"  Samples: {n}")

# Run on all prediction files
pred_files = sorted([
    f"{PRED_DIR}/{f}" for f in os.listdir(PRED_DIR)
    if f.endswith(".jsonl")
])

print("=" * 50)
print("ACCURACY RESULTS")
print("=" * 50)
for pf in pred_files:
    quick_accuracy(pf)

print("\nPaper baseline (LLaMA2 zero-shot): 4.27% overall")
print("Paper best (ensemble):             63.8% overall")

ACCURACY RESULTS

--- checkpoint-1604.jsonl ---
  Overall accuracy:         40.80%
  Event type accuracy:      61.20%
  Informativeness accuracy: 80.40%
  Humanitarian accuracy:    58.60%
  Samples: 500

--- checkpoint-3208.jsonl ---
  Overall accuracy:         57.00%
  Event type accuracy:      78.00%
  Informativeness accuracy: 85.60%
  Humanitarian accuracy:    66.60%
  Samples: 500

--- checkpoint-4812.jsonl ---
  Overall accuracy:         57.60%
  Event type accuracy:      77.80%
  Informativeness accuracy: 81.80%
  Humanitarian accuracy:    66.80%
  Samples: 500

--- checkpoint-6416.jsonl ---
  Overall accuracy:         58.80%
  Event type accuracy:      81.00%
  Informativeness accuracy: 81.00%
  Humanitarian accuracy:    66.60%
  Samples: 500

--- zero_shot.jsonl ---
  Overall accuracy:         8.60%
  Event type accuracy:      35.60%
  Informativeness accuracy: 72.20%
  Humanitarian accuracy:    36.60%
  Samples: 500

Paper baseline (LLaMA2 zero-shot): 4.27% overall
Paper best